# Lab 22 solution: worked answers to the challenge

The worked answers to the challenge of `22_process_reengineering.ipynb`. The setup repeats the definitions the
three tasks use (the lunch minutes, the answers and `route`, the scenario and `estimate`), so this file runs on its
own. It reads no files and calls no model, so it needs no setup cell.

The lunch minutes after the handover, from Part A:

In [1]:
lunch_before = 5 + 3 + 2 + 2   # collect, check, place, pay
lunch_after = {"collect": 0, "check": 0, "place": 0, "pay": 1}

The answers and `route`, from Part D:

In [2]:
answers = {
    #           Q1 steps written down, Q2 evidence a tool can read, Q3 a mistake can be undone cheaply
    "log":     (True,  True,  True),
    "check":   (True,  True,  True),
    "history": (False, True,  True),
    "decide":  (True,  True,  True),
    "reply":   (True,  True,  True),
    "refund":  (True,  True,  False),
}


def route(q1, q2, q3):
    path = "workflow" if q1 else "agent"
    needs = []
    if not q2:
        needs.append("fix the record first")
    if not q3:
        needs.append("a person approves")
    return path, needs

The scenario and `estimate`, from Part G, with the three numbers the notebook took from earlier parts written in:
18 minutes before, 2 of handling after and 2 of review.

In [3]:
scenario = {
    "cases_per_month": 300,
    "minutes_before": 18,
    "minutes_after": 2,
    "review_minutes": 2,
    "rework_share": 0.10,              # one case in ten comes back ...
    "rework_minutes": 12,              # ... and takes 12 more minutes
    "wage_usd_per_hour": 30,
    "model_usd_per_case": 0.01,        # the agent's searches and the drafted reply
    "running_usd_per_month": 200,      # hosting, logs and the review queue
    "build_usd": 12_000,               # building it, once
    "build_months": 12,                # the build spread over a year
}


def estimate(s):
    saved_per_case = s["minutes_before"] - s["minutes_after"] - s["review_minutes"] - s["rework_share"] * s["rework_minutes"]
    freed_minutes = s["cases_per_month"] * saved_per_case
    freed_value = freed_minutes / 60 * s["wage_usd_per_hour"]
    system_cost = (s["cases_per_month"] * s["model_usd_per_case"] + s["running_usd_per_month"]
                   + s["build_usd"] / s["build_months"])
    return freed_minutes, freed_value - system_cost


print(estimate(scenario))

(3840.0, 717.0)


The base result, as in Part G: 3,840 minutes freed and 717 dollars net.

## Task 1: read back only the orders that need it

Three orders of twelve have an allergy or a comment, so the read-back takes 3 minutes instead of 12.

In [4]:
read_back = 3 * 1   # three flagged orders, about a minute each
lunch_now = sum(lunch_after.values()) + read_back
print("Before:", lunch_before, "min | after the handover, reading back only the flagged orders:", lunch_now, "min")

Before: 12 min | after the handover, reading back only the flagged orders: 4 min


**4 minutes against 12.** The handover's saving is back, and the orders with an allergy are still read back by a
person. Nothing new was automated: we changed which orders get the check, which is a change to the process.

## Task 2: the photo is missing

We copy the `check` step's answers and set Q2 to `False`. A tuple cannot be changed in place, so we build a new one
from the old values.

In [5]:
q1, q2, q3 = answers["check"]
no_photo = (q1, False, q3)
print("check, no photo:", no_photo)
print(route(*no_photo))

check, no photo: (True, False, True)
('workflow', ['fix the record first'])


`('workflow', ['fix the record first'])`. The path is still known: whether there is a photo or not, the next step is
written down. What is missing is the evidence, so the step's first job is to ask the customer for the photo, and the
decision and the refund wait until it arrives. This is why Q1 and Q2 are kept apart: a missing record is not a
reason for an agent, it is a reason to fix the record.

## Task 3: find where the redesign stops paying

In [6]:
for minutes in range(2, 11):
    trial = scenario.copy()
    trial["review_minutes"] = minutes
    freed_minutes, net = estimate(trial)
    print(f"review {minutes:2} min | freed {freed_minutes:5,.0f} min a month | net {net:+6,.0f} USD a month")

review  2 min | freed 3,840 min a month | net   +717 USD a month
review  3 min | freed 3,540 min a month | net   +567 USD a month
review  4 min | freed 3,240 min a month | net   +417 USD a month
review  5 min | freed 2,940 min a month | net   +267 USD a month
review  6 min | freed 2,640 min a month | net   +117 USD a month
review  7 min | freed 2,340 min a month | net    -33 USD a month
review  8 min | freed 2,040 min a month | net   -183 USD a month
review  9 min | freed 1,740 min a month | net   -333 USD a month
review 10 min | freed 1,440 min a month | net   -483 USD a month


Each extra minute of review costs 300 minutes a month, which is 150 dollars at 30 dollars an hour, so the net falls
by 150 dollars a row. It is **+117 dollars at 6 minutes and −33 dollars at 7**: the redesign stops paying between 6
and 7 minutes of review a case. Every row still frees time, 1,440 minutes a month even at 10 minutes of review. Freed
minutes alone do not show whether the redesign is worth building. The net does.